# BRISC Dataset Integrity Checks

This notebook verifies dataset metadata, manifest integrity, duplicates, and possible train/test leakage before preprocessing.

In [1]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "brisc2025"
CLASSIFICATION_ROOT = DATASET_ROOT / "classification_task"
SEGMENTATION_ROOT = DATASET_ROOT / "segmentation_task"

MANIFEST_CSV = DATASET_ROOT / "manifest.csv"
MANIFEST_JSON = DATASET_ROOT / "manifest.json"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)

print("\nPath checks:")
print("Dataset:", DATASET_ROOT.exists())
print("Classification:", CLASSIFICATION_ROOT.exists())
print("Segmentation:", SEGMENTATION_ROOT.exists())
print("Manifest CSV:", MANIFEST_CSV.exists())
print("Manifest JSON:", MANIFEST_JSON.exists())

Project root: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI
Dataset root: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\raw\brisc2025

Path checks:
Dataset: True
Classification: True
Segmentation: True
Manifest CSV: True
Manifest JSON: True


## 1. Manifest Inspection

import pandas as pd

manifest_df = pd.read_csv(MANIFEST_CSV)

print("Manifest shape:", manifest_df.shape)

print("\nManifest columns:")
for i, column in enumerate(manifest_df.columns, start=1):
    print(f"{i}. {column}")

print("\nFirst 5 rows:")
display(manifest_df.head())

## 2. Manifest SHA-256 Verification

In [3]:
import hashlib


def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def read_expected_hash(sha_file):
    with open(sha_file, "r", encoding="utf-8") as f:
        content = f.read().strip()

    # Works whether the sidecar contains only the hash
    # or "hash filename"
    return content.split()[0]


files_to_verify = [
    (
        DATASET_ROOT / "manifest.csv",
        DATASET_ROOT / "manifest.csv.sha256"
    ),
    (
        DATASET_ROOT / "manifest.json",
        DATASET_ROOT / "manifest.json.sha256"
    )
]


for data_file, sha_file in files_to_verify:
    calculated = calculate_sha256(data_file)
    expected = read_expected_hash(sha_file)

    print(f"\n{data_file.name}")
    print("-" * 45)
    print("Expected :", expected)
    print("Calculated:", calculated)
    print(
        "Status    :",
        "MATCH" if calculated.lower() == expected.lower()
        else "MISMATCH"
    )


manifest.csv
---------------------------------------------
Expected : 4eaa0c3b88ce1fd246d90382c1dcd1010cc126fff1bd3a4a415159661ca8b605
Calculated: 4eaa0c3b88ce1fd246d90382c1dcd1010cc126fff1bd3a4a415159661ca8b605
Status    : MATCH

manifest.json
---------------------------------------------
Expected : ada767e102e9e7ee788b50188712dfe19ba825c250c362d831b560902c249e3a
Calculated: ada767e102e9e7ee788b50188712dfe19ba825c250c362d831b560902c249e3a
Status    : MATCH


## 3. Manifest-to-Filesystem Consistency

In [5]:
print("Manifest shape:", manifest_df.shape)

print("\nRows by task:")
print(manifest_df["task"].value_counts())

print("\nRows by split:")
print(manifest_df["split"].value_counts())

print("\nRows by task and split:")
print(manifest_df.groupby(["task", "split"]).size())

print("\nMask flag counts:")
print(manifest_df["is_mask"].value_counts())

Manifest shape: (15586, 16)

Rows by task:
task
segmentation      9586
classification    6000
Name: count, dtype: int64

Rows by split:
split
train    12866
test      2720
Name: count, dtype: int64

Rows by task and split:
task            split
classification  test     1000
                train    5000
segmentation    test     1720
                train    7866
dtype: int64

Mask flag counts:
is_mask
False    10793
True      4793
Name: count, dtype: int64


In [6]:
# Check whether every manifest relative_path exists physically

missing_manifest_files = []

for rel_path in manifest_df["relative_path"]:
    file_path = DATASET_ROOT / Path(rel_path)

    if not file_path.exists():
        missing_manifest_files.append(rel_path)

print("Manifest rows checked:", len(manifest_df))
print("Manifest paths missing on disk:", len(missing_manifest_files))

if len(missing_manifest_files) == 0:
    print("Filesystem status: ALL MANIFEST FILES FOUND")
else:
    print("Filesystem status: MISSING FILES FOUND")
    print("\nFirst few missing paths:")
    for item in missing_manifest_files[:10]:
        print(item)

Manifest rows checked: 15586
Manifest paths missing on disk: 0
Filesystem status: ALL MANIFEST FILES FOUND


## 4. Filesystem-to-Manifest Reverse Consistency

In [7]:
# Collect every actual data file inside the two task directories

actual_data_files = []

for task_root in [CLASSIFICATION_ROOT, SEGMENTATION_ROOT]:
    actual_data_files.extend(
        f for f in task_root.rglob("*") if f.is_file()
    )

# Normalize all paths relative to the BRISC dataset root
actual_relative_paths = {
    f.relative_to(DATASET_ROOT).as_posix()
    for f in actual_data_files
}

manifest_relative_paths = {
    Path(p).as_posix()
    for p in manifest_df["relative_path"]
}

extra_files = sorted(
    actual_relative_paths - manifest_relative_paths
)

missing_files = sorted(
    manifest_relative_paths - actual_relative_paths
)

print("Actual data files on disk :", len(actual_relative_paths))
print("Manifest data records     :", len(manifest_relative_paths))

print("\nFiles on disk but NOT in manifest:", len(extra_files))
print("Manifest files missing from disk :", len(missing_files))

if not extra_files and not missing_files:
    print("\nReverse consistency status: PERFECT MATCH")
else:
    print("\nReverse consistency status: MISMATCH FOUND")

    if extra_files:
        print("\nFirst few extra files:")
        for item in extra_files[:10]:
            print(item)

    if missing_files:
        print("\nFirst few missing files:")
        for item in missing_files[:10]:
            print(item)

Actual data files on disk : 15586
Manifest data records     : 15586

Files on disk but NOT in manifest: 0
Manifest files missing from disk : 0

Reverse consistency status: PERFECT MATCH


## 5. Manifest Metadata Validation

In [8]:
from PIL import Image

dimension_mismatches = []
size_mismatches = []

for _, row in manifest_df.iterrows():
    file_path = DATASET_ROOT / Path(row["relative_path"])

    # Actual file size in bytes
    actual_file_size = file_path.stat().st_size

    if actual_file_size != int(row["file_size_bytes"]):
        size_mismatches.append({
            "filename": row["filename"],
            "manifest_size": int(row["file_size_bytes"]),
            "actual_size": actual_file_size
        })

    # Actual image dimensions
    with Image.open(file_path) as img:
        actual_width, actual_height = img.size

    if (
        actual_width != int(row["width"])
        or actual_height != int(row["height"])
    ):
        dimension_mismatches.append({
            "filename": row["filename"],
            "manifest_dimensions": (
                int(row["width"]),
                int(row["height"])
            ),
            "actual_dimensions": (
                actual_width,
                actual_height
            )
        })


print("Records checked:", len(manifest_df))

print("\nDimension mismatches:", len(dimension_mismatches))
print("File-size mismatches :", len(size_mismatches))

if not dimension_mismatches and not size_mismatches:
    print("\nMetadata validation status: PERFECT MATCH")
else:
    print("\nMetadata validation status: MISMATCH FOUND")

    if dimension_mismatches:
        print("\nFirst few dimension mismatches:")
        for item in dimension_mismatches[:10]:
            print(item)

    if size_mismatches:
        print("\nFirst few size mismatches:")
        for item in size_mismatches[:10]:
            print(item)

Records checked: 15586

Dimension mismatches: 0
File-size mismatches : 0

Metadata validation status: PERFECT MATCH


## 6. Per-File SHA-256 Verification

In [9]:
import hashlib

def file_sha256(file_path):
    sha = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)

    return sha.hexdigest()


hash_mismatches = []

for _, row in manifest_df.iterrows():
    file_path = DATASET_ROOT / Path(row["relative_path"])

    calculated_hash = file_sha256(file_path)
    expected_hash = str(row["sha256"]).lower()

    if calculated_hash.lower() != expected_hash:
        hash_mismatches.append({
            "filename": row["filename"],
            "expected": expected_hash,
            "calculated": calculated_hash
        })


print("Files hashed:", len(manifest_df))
print("SHA-256 mismatches:", len(hash_mismatches))

if not hash_mismatches:
    print("Per-file integrity status: ALL FILES VERIFIED")
else:
    print("Per-file integrity status: MISMATCH FOUND")

    print("\nFirst few mismatches:")
    for item in hash_mismatches[:10]:
        print(item)

Files hashed: 15586
SHA-256 mismatches: 0
Per-file integrity status: ALL FILES VERIFIED


## 7. Exact Duplicate and Train–Test Leakage Checks

In [10]:
# Classification records only
classification_df = manifest_df[
    manifest_df["task"] == "classification"
].copy()

print("Classification records:", len(classification_df))


# --------------------------------------------------
# Exact duplicate hashes anywhere in classification
# --------------------------------------------------

duplicate_classification = classification_df[
    classification_df.duplicated(
        subset="sha256",
        keep=False
    )
].sort_values("sha256")

duplicate_groups = (
    duplicate_classification["sha256"].nunique()
    if not duplicate_classification.empty
    else 0
)

print("\nExact duplicate groups:", duplicate_groups)
print(
    "Rows involved in exact duplicates:",
    len(duplicate_classification)
)


# --------------------------------------------------
# Exact duplicates crossing TRAIN and TEST
# --------------------------------------------------

cross_split_hashes = []

for sha, group in classification_df.groupby("sha256"):
    if group["split"].nunique() > 1:
        cross_split_hashes.append(sha)

cross_split_duplicates = classification_df[
    classification_df["sha256"].isin(cross_split_hashes)
].sort_values(["sha256", "split"])

print(
    "\nExact hashes appearing in BOTH train and test:",
    len(cross_split_hashes)
)

print(
    "Rows involved in train/test exact overlap:",
    len(cross_split_duplicates)
)


# --------------------------------------------------
# Same exact image assigned to different classes
# --------------------------------------------------

cross_class_hashes = []

for sha, group in classification_df.groupby("sha256"):
    if group["tumor_label"].nunique() > 1:
        cross_class_hashes.append(sha)

cross_class_duplicates = classification_df[
    classification_df["sha256"].isin(cross_class_hashes)
].sort_values(["sha256", "tumor_label"])

print(
    "\nExact hashes assigned to multiple classes:",
    len(cross_class_hashes)
)

if len(cross_split_hashes) == 0:
    print("\nTrain/Test exact-leakage status: NO EXACT OVERLAP")
else:
    print("\nTrain/Test exact-leakage status: EXACT OVERLAP FOUND")

if len(cross_class_hashes) == 0:
    print("Cross-class conflict status: NONE FOUND")
else:
    print("Cross-class conflict status: CONFLICT FOUND")

Classification records: 6000

Exact duplicate groups: 46
Rows involved in exact duplicates: 96

Exact hashes appearing in BOTH train and test: 7
Rows involved in train/test exact overlap: 16

Exact hashes assigned to multiple classes: 0

Train/Test exact-leakage status: EXACT OVERLAP FOUND
Cross-class conflict status: NONE FOUND


### 7.1 Inspection of Exact Train–Test Overlaps

In [11]:
# Show every classification record involved in exact train/test overlap

columns_to_show = [
    "filename",
    "split",
    "tumor_label",
    "plane_label",
    "sequence",
    "width",
    "height",
    "sha256",
    "relative_path"
]

cross_split_details = cross_split_duplicates[
    columns_to_show
].copy()

display(cross_split_details)

,filename,split,tumor_label,plane_label,sequence,width,height,sha256,relative_path
826,brisc2025_test_00827_pi_co_t1.jpg,test,pituitary,coronal,T1,512,512,02799f7863d49de182357e72179f3cc4e65e70099355f2...,classification_task\test\pituitary\brisc2025_t...
5193,brisc2025_train_04194_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,02799f7863d49de182357e72179f3cc4e65e70099355f2...,classification_task\train\pituitary\brisc2025_...
750,brisc2025_test_00751_pi_ax_t1.jpg,test,pituitary,axial,T1,512,512,08690b76457b2b44d58682cece62ec951ef6d30d935ce0...,classification_task\test\pituitary\brisc2025_t...
4748,brisc2025_train_03749_pi_ax_t1.jpg,train,pituitary,axial,T1,512,512,08690b76457b2b44d58682cece62ec951ef6d30d935ce0...,classification_task\train\pituitary\brisc2025_...
736,brisc2025_test_00737_pi_ax_t1.jpg,test,pituitary,axial,T1,512,512,39f4f7b252d7809d8d3d2e4521607de2195bf0f66c0592...,classification_task\test\pituitary\brisc2025_t...
4704,brisc2025_train_03705_pi_ax_t1.jpg,train,pituitary,axial,T1,512,512,39f4f7b252d7809d8d3d2e4521607de2195bf0f66c0592...,classification_task\train\pituitary\brisc2025_...
824,brisc2025_test_00825_pi_co_t1.jpg,test,pituitary,coronal,T1,512,512,3e802dd22f483ce4d442fe2c7f1890c1c810ec8c1131b3...,classification_task\test\pituitary\brisc2025_t...
5175,brisc2025_train_04176_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,3e802dd22f483ce4d442fe2c7f1890c1c810ec8c1131b3...,classification_task\train\pituitary\brisc2025_...
5176,brisc2025_train_04177_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,3e802dd22f483ce4d442fe2c7f1890c1c810ec8c1131b3...,classification_task\train\pituitary\brisc2025_...
5177,brisc2025_train_04178_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,3e802dd22f483ce4d442fe2c7f1890c1c810ec8c1131b3...,classification_task\train\pituitary\brisc2025_...


In [12]:
# Summarize each duplicated hash group individually

for group_number, (sha, group) in enumerate(
    cross_split_duplicates.groupby("sha256"),
    start=1
):
    print(f"\nDUPLICATE GROUP {group_number}")
    print("=" * 70)
    print("SHA-256:", sha)
    print("Number of records:", len(group))

    display(
        group[
            [
                "filename",
                "split",
                "tumor_label",
                "plane_label",
                "sequence",
                "width",
                "height",
                "relative_path"
            ]
        ]
    )


DUPLICATE GROUP 1
SHA-256: 02799f7863d49de182357e72179f3cc4e65e70099355f2d5aa2e39d356e492d9
Number of records: 2


,filename,split,tumor_label,plane_label,sequence,width,height,relative_path
826,brisc2025_test_00827_pi_co_t1.jpg,test,pituitary,coronal,T1,512,512,classification_task\test\pituitary\brisc2025_t...
5193,brisc2025_train_04194_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,classification_task\train\pituitary\brisc2025_...



DUPLICATE GROUP 2
SHA-256: 08690b76457b2b44d58682cece62ec951ef6d30d935ce07df852e97d44434680
Number of records: 2


,filename,split,tumor_label,plane_label,sequence,width,height,relative_path
750,brisc2025_test_00751_pi_ax_t1.jpg,test,pituitary,axial,T1,512,512,classification_task\test\pituitary\brisc2025_t...
4748,brisc2025_train_03749_pi_ax_t1.jpg,train,pituitary,axial,T1,512,512,classification_task\train\pituitary\brisc2025_...



DUPLICATE GROUP 3
SHA-256: 39f4f7b252d7809d8d3d2e4521607de2195bf0f66c0592587b1cebd632aac640
Number of records: 2


,filename,split,tumor_label,plane_label,sequence,width,height,relative_path
736,brisc2025_test_00737_pi_ax_t1.jpg,test,pituitary,axial,T1,512,512,classification_task\test\pituitary\brisc2025_t...
4704,brisc2025_train_03705_pi_ax_t1.jpg,train,pituitary,axial,T1,512,512,classification_task\train\pituitary\brisc2025_...


... (4 outputs left out)


DUPLICATE GROUP 6
SHA-256: b5e17076d1427e41a618f92bc72100c1be4df26608223512aac51c911ac1eb88
Number of records: 2


,filename,split,tumor_label,plane_label,sequence,width,height,relative_path
833,brisc2025_test_00834_pi_co_t1.jpg,test,pituitary,coronal,T1,512,512,classification_task\test\pituitary\brisc2025_t...
5217,brisc2025_train_04218_pi_co_t1.jpg,train,pituitary,coronal,T1,512,512,classification_task\train\pituitary\brisc2025_...



DUPLICATE GROUP 7
SHA-256: f3d00bbb942117c2fb62466414d3c466ca283b678200fe87424e4128b80b2dba
Number of records: 2


,filename,split,tumor_label,plane_label,sequence,width,height,relative_path
351,brisc2025_test_00352_me_ax_t1.jpg,test,meningioma,axial,T1,201,251,classification_task\test\meningioma\brisc2025_...
2303,brisc2025_train_01304_me_ax_t1.jpg,train,meningioma,axial,T1,201,251,classification_task\train\meningioma\brisc2025...


for split in ["train", "test"]:
    split_df = classification_df[
        classification_df["split"] == split
    ]

    duplicated = split_df[
        split_df.duplicated(
            subset="sha256",
            keep=False
        )
    ]

    groups = (
        duplicated["sha256"].nunique()
        if not duplicated.empty
        else 0
    )

    print(f"\n{split.upper()} SPLIT")
    print("-" * 45)
    print("Images:", len(split_df))
    print("Exact duplicate groups:", groups)
    print("Rows involved:", len(duplicated))

### Exact-Duplicate Audit Finding

The classification dataset contains 46 SHA-256 duplicate groups involving
96 records. Seven unique image hashes occur across both the official
training and testing splits, involving 9 training records and 7 testing
records (16 records total).

No exact duplicate hash was assigned to more than one tumor class.

The raw BRISC dataset will remain unchanged. Cross-split duplicates will
be handled through a reproducible exclusion/split protocol before model
training.

### 7.2 Saving Exact-Duplicate Audit

In [14]:
INTERIM_ROOT = PROJECT_ROOT / "data" / "interim"
INTERIM_ROOT.mkdir(parents=True, exist_ok=True)

# Save all exact classification duplicates
duplicate_classification.to_csv(
    INTERIM_ROOT / "classification_exact_duplicates.csv",
    index=False
)

# Save only train/test crossing duplicates
cross_split_duplicates.to_csv(
    INTERIM_ROOT / "classification_cross_split_exact_duplicates.csv",
    index=False
)

print("Saved:")
print(
    INTERIM_ROOT /
    "classification_exact_duplicates.csv"
)
print(
    INTERIM_ROOT /
    "classification_cross_split_exact_duplicates.csv"
)

Saved:
C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\interim\classification_exact_duplicates.csv
C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\interim\classification_cross_split_exact_duplicates.csv


## 8. Segmentation MRI Exact-Leakage Check

In [15]:
# Only segmentation MRI images, excluding masks
segmentation_mri_df = manifest_df[
    (manifest_df["task"] == "segmentation") &
    (manifest_df["is_mask"] == False)
].copy()

print("Segmentation MRI records:", len(segmentation_mri_df))

seg_cross_split_hashes = []

for sha, group in segmentation_mri_df.groupby("sha256"):
    if group["split"].nunique() > 1:
        seg_cross_split_hashes.append(sha)

seg_cross_split_duplicates = segmentation_mri_df[
    segmentation_mri_df["sha256"].isin(seg_cross_split_hashes)
].sort_values(["sha256", "split"])

print(
    "\nExact MRI hashes appearing in BOTH segmentation train and test:",
    len(seg_cross_split_hashes)
)

print(
    "Rows involved:",
    len(seg_cross_split_duplicates)
)

if len(seg_cross_split_hashes) == 0:
    print(
        "\nSegmentation train/test exact-leakage status: "
        "NO EXACT OVERLAP"
    )
else:
    print(
        "\nSegmentation train/test exact-leakage status: "
        "EXACT OVERLAP FOUND"
    )

Segmentation MRI records: 4793

Exact MRI hashes appearing in BOTH segmentation train and test: 7
Rows involved: 16

Segmentation train/test exact-leakage status: EXACT OVERLAP FOUND


### 8.1 Cross-Task Comparison of Exact Train–Test Overlaps

In [16]:
classification_overlap_hashes = set(cross_split_hashes)
segmentation_overlap_hashes = set(seg_cross_split_hashes)

common_overlap_hashes = (
    classification_overlap_hashes &
    segmentation_overlap_hashes
)

classification_only = (
    classification_overlap_hashes -
    segmentation_overlap_hashes
)

segmentation_only = (
    segmentation_overlap_hashes -
    classification_overlap_hashes
)

print("Classification cross-split hashes:",
      len(classification_overlap_hashes))

print("Segmentation cross-split hashes:",
      len(segmentation_overlap_hashes))

print("\nCommon to BOTH tasks:",
      len(common_overlap_hashes))

print("Classification-only overlap hashes:",
      len(classification_only))

print("Segmentation-only overlap hashes:",
      len(segmentation_only))

if (
    classification_overlap_hashes ==
    segmentation_overlap_hashes
):
    print(
        "\nResult: The exact same cross-split MRI "
        "overlaps occur in both tasks."
    )
else:
    print(
        "\nResult: The cross-split overlap sets "
        "are not identical."
    )

Classification cross-split hashes: 7
Segmentation cross-split hashes: 7

Common to BOTH tasks: 7
Classification-only overlap hashes: 0
Segmentation-only overlap hashes: 0

Result: The exact same cross-split MRI overlaps occur in both tasks.


### 8.2 Segmentation Mask Exact-Overlap Check

In [17]:
segmentation_mask_df = manifest_df[
    (manifest_df["task"] == "segmentation") &
    (manifest_df["is_mask"] == True)
].copy()

mask_cross_split_hashes = []

for sha, group in segmentation_mask_df.groupby("sha256"):
    if group["split"].nunique() > 1:
        mask_cross_split_hashes.append(sha)

mask_cross_split_duplicates = segmentation_mask_df[
    segmentation_mask_df["sha256"].isin(
        mask_cross_split_hashes
    )
].sort_values(["sha256", "split"])

print("Segmentation mask records:",
      len(segmentation_mask_df))

print(
    "\nExact mask hashes appearing in BOTH train and test:",
    len(mask_cross_split_hashes)
)

print(
    "Rows involved:",
    len(mask_cross_split_duplicates)
)

Segmentation mask records: 4793

Exact mask hashes appearing in BOTH train and test: 0
Rows involved: 0


### 8.3 Pixel-Level Mask Comparison for Cross-Split Duplicate MRIs

In [18]:
import numpy as np
from PIL import Image

# Segmentation MRI records involved in the known cross-split overlap
overlap_mri_records = segmentation_mri_df[
    segmentation_mri_df["sha256"].isin(seg_cross_split_hashes)
].copy()

# All segmentation mask records
mask_df = manifest_df[
    (manifest_df["task"] == "segmentation") &
    (manifest_df["is_mask"] == True)
].copy()


def find_linked_mask(image_filename, split):
    matches = mask_df[
        (mask_df["linked_image"] == image_filename) &
        (mask_df["split"] == split)
    ]

    if len(matches) == 1:
        return matches.iloc[0]

    return None


comparison_results = []

for sha, group in overlap_mri_records.groupby("sha256"):

    train_rows = group[group["split"] == "train"]
    test_rows = group[group["split"] == "test"]

    for _, train_img in train_rows.iterrows():
        for _, test_img in test_rows.iterrows():

            train_mask = find_linked_mask(
                train_img["filename"], "train"
            )

            test_mask = find_linked_mask(
                test_img["filename"], "test"
            )

            if train_mask is None or test_mask is None:
                comparison_results.append({
                    "train_image": train_img["filename"],
                    "test_image": test_img["filename"],
                    "status": "MASK LINK NOT FOUND"
                })
                continue

            train_mask_path = (
                DATASET_ROOT /
                Path(train_mask["relative_path"])
            )

            test_mask_path = (
                DATASET_ROOT /
                Path(test_mask["relative_path"])
            )

            with Image.open(train_mask_path) as im1:
                arr1 = np.array(im1)

            with Image.open(test_mask_path) as im2:
                arr2 = np.array(im2)

            same_shape = arr1.shape == arr2.shape

            pixel_equal = (
                same_shape and
                np.array_equal(arr1, arr2)
            )

            different_pixels = (
                np.count_nonzero(arr1 != arr2)
                if same_shape
                else None
            )

            comparison_results.append({
                "train_image": train_img["filename"],
                "test_image": test_img["filename"],
                "train_mask": train_mask["filename"],
                "test_mask": test_mask["filename"],
                "same_shape": same_shape,
                "pixel_identical": pixel_equal,
                "different_pixels": different_pixels
            })


mask_comparison_df = pd.DataFrame(comparison_results)

display(mask_comparison_df)

print("\nSUMMARY")
print("-" * 45)

if "pixel_identical" in mask_comparison_df.columns:
    print(
        "Mask pairs compared:",
        mask_comparison_df["pixel_identical"].notna().sum()
    )

    print(
        "Pixel-identical masks:",
        (mask_comparison_df["pixel_identical"] == True).sum()
    )

    print(
        "Pixel-different masks:",
        (mask_comparison_df["pixel_identical"] == False).sum()
    )

,train_image,test_image,status
0,brisc2025_train_04194_pi_co_t1.jpg,brisc2025_test_00827_pi_co_t1.jpg,MASK LINK NOT FOUND
1,brisc2025_train_03749_pi_ax_t1.jpg,brisc2025_test_00751_pi_ax_t1.jpg,MASK LINK NOT FOUND
2,brisc2025_train_03705_pi_ax_t1.jpg,brisc2025_test_00737_pi_ax_t1.jpg,MASK LINK NOT FOUND
3,brisc2025_train_04176_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,MASK LINK NOT FOUND
4,brisc2025_train_04177_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,MASK LINK NOT FOUND
5,brisc2025_train_04178_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,MASK LINK NOT FOUND
6,brisc2025_train_01210_me_ax_t1.jpg,brisc2025_test_00344_me_ax_t1.jpg,MASK LINK NOT FOUND
7,brisc2025_train_04218_pi_co_t1.jpg,brisc2025_test_00834_pi_co_t1.jpg,MASK LINK NOT FOUND
8,brisc2025_train_01304_me_ax_t1.jpg,brisc2025_test_00352_me_ax_t1.jpg,MASK LINK NOT FOUND



SUMMARY
---------------------------------------------


### 8.3a Inspect Mask Linkage Format

In [19]:
# Inspect how segmentation masks reference their corresponding MRI images

mask_link_preview = mask_df[
    [
        "filename",
        "split",
        "tumor_label",
        "plane_label",
        "linked_image",
        "relative_path"
    ]
].head(15)

display(mask_link_preview)

print("\nExample linked_image values:")
for value in mask_df["linked_image"].dropna().head(10):
    print(repr(value))

,filename,split,tumor_label,plane_label,linked_image,relative_path
6860,brisc2025_test_00001_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6861,brisc2025_test_00002_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6862,brisc2025_test_00003_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6863,brisc2025_test_00004_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6864,brisc2025_test_00005_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6865,brisc2025_test_00006_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6866,brisc2025_test_00007_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6867,brisc2025_test_00008_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6868,brisc2025_test_00009_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
6869,brisc2025_test_00010_gl_ax_t1.png,test,glioma,axial,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...



Example linked_image values:
'segmentation_task\\test\\images\\brisc2025_test_00001_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00002_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00003_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00004_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00005_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00006_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00007_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00008_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00009_gl_ax_t1.jpg'
'segmentation_task\\test\\images\\brisc2025_test_00010_gl_ax_t1.jpg'


In [21]:
# Search the manifest for one of our known duplicate MRI identifiers

search_term = "04194"

matches = manifest_df[
    manifest_df.astype(str).apply(
        lambda row: row.str.contains(
            search_term,
            case=False,
            na=False
        ).any(),
        axis=1
    )
]

display(
    matches[
        [
            "filename",
            "task",
            "split",
            "is_mask",
            "linked_image",
            "relative_path"
        ]
    ]
)

,filename,task,split,is_mask,linked_image,relative_path
5193,brisc2025_train_04194_pi_co_t1.jpg,classification,train,False,NaN,classification_task\train\pituitary\brisc2025_...
7571,brisc2025_test_00852_pi_co_t1.png,segmentation,test,True,segmentation_task\test\images\brisc2025_test_0...,segmentation_task\test\masks\brisc2025_test_00...
10846,brisc2025_train_04194_pi_co_t1.jpg,segmentation,train,False,NaN,segmentation_task\train\images\brisc2025_train...
14779,brisc2025_train_04194_pi_co_t1.png,segmentation,train,True,segmentation_task\train\images\brisc2025_train...,segmentation_task\train\masks\brisc2025_train_...


### 8.3b Corrected Mask-Link Verification

In [22]:
from pathlib import PureWindowsPath

mask_df = mask_df.copy()

# Extract just the MRI filename from BRISC's linked_image path
mask_df["linked_image_filename"] = mask_df["linked_image"].apply(
    lambda x: PureWindowsPath(x).name
    if pd.notna(x)
    else None
)

known_images = [
    "brisc2025_train_04194_pi_co_t1.jpg",
    "brisc2025_test_00827_pi_co_t1.jpg"
]

known_masks = mask_df[
    mask_df["linked_image_filename"].isin(known_images)
]

display(
    known_masks[
        [
            "filename",
            "split",
            "tumor_label",
            "plane_label",
            "linked_image_filename",
            "relative_path"
        ]
    ]
)

,filename,split,tumor_label,plane_label,linked_image_filename,relative_path
7546,brisc2025_test_00827_pi_co_t1.png,test,pituitary,coronal,brisc2025_test_00827_pi_co_t1.jpg,segmentation_task\test\masks\brisc2025_test_00...
14779,brisc2025_train_04194_pi_co_t1.png,train,pituitary,coronal,brisc2025_train_04194_pi_co_t1.jpg,segmentation_task\train\masks\brisc2025_train_...


In [23]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path


def find_linked_mask(image_filename, split):
    matches = mask_df[
        (mask_df["linked_image_filename"] == image_filename) &
        (mask_df["split"] == split)
    ]

    if len(matches) == 1:
        return matches.iloc[0]

    return None


overlap_mri_records = segmentation_mri_df[
    segmentation_mri_df["sha256"].isin(seg_cross_split_hashes)
].copy()

comparison_results = []

for sha, group in overlap_mri_records.groupby("sha256"):

    train_rows = group[group["split"] == "train"]
    test_rows = group[group["split"] == "test"]

    for _, train_img in train_rows.iterrows():
        for _, test_img in test_rows.iterrows():

            train_mask = find_linked_mask(
                train_img["filename"], "train"
            )

            test_mask = find_linked_mask(
                test_img["filename"], "test"
            )

            if train_mask is None or test_mask is None:
                comparison_results.append({
                    "train_image": train_img["filename"],
                    "test_image": test_img["filename"],
                    "status": "MASK LINK NOT FOUND"
                })
                continue

            train_mask_path = (
                DATASET_ROOT /
                Path(train_mask["relative_path"])
            )

            test_mask_path = (
                DATASET_ROOT /
                Path(test_mask["relative_path"])
            )

            with Image.open(train_mask_path) as im1:
                arr1 = np.array(im1)

            with Image.open(test_mask_path) as im2:
                arr2 = np.array(im2)

            same_shape = arr1.shape == arr2.shape

            pixel_identical = (
                same_shape and
                np.array_equal(arr1, arr2)
            )

            different_pixels = (
                int(np.count_nonzero(arr1 != arr2))
                if same_shape
                else None
            )

            comparison_results.append({
                "train_image": train_img["filename"],
                "test_image": test_img["filename"],
                "train_mask": train_mask["filename"],
                "test_mask": test_mask["filename"],
                "same_shape": same_shape,
                "pixel_identical": pixel_identical,
                "different_pixels": different_pixels,
                "status": "OK"
            })


mask_comparison_df = pd.DataFrame(comparison_results)

display(mask_comparison_df)

print("\nSUMMARY")
print("-" * 50)
print("Comparisons attempted:", len(mask_comparison_df))
print(
    "Mask links successfully found:",
    (mask_comparison_df["status"] == "OK").sum()
)

if "pixel_identical" in mask_comparison_df.columns:
    print(
        "Pixel-identical mask pairs:",
        (mask_comparison_df["pixel_identical"] == True).sum()
    )
    print(
        "Pixel-different mask pairs:",
        (mask_comparison_df["pixel_identical"] == False).sum()
    )

,train_image,test_image,train_mask,test_mask,same_shape,pixel_identical,different_pixels,status
0,brisc2025_train_04194_pi_co_t1.jpg,brisc2025_test_00827_pi_co_t1.jpg,brisc2025_train_04194_pi_co_t1.png,brisc2025_test_00827_pi_co_t1.png,True,False,789,OK
1,brisc2025_train_03749_pi_ax_t1.jpg,brisc2025_test_00751_pi_ax_t1.jpg,brisc2025_train_03749_pi_ax_t1.png,brisc2025_test_00751_pi_ax_t1.png,True,False,1475,OK
2,brisc2025_train_03705_pi_ax_t1.jpg,brisc2025_test_00737_pi_ax_t1.jpg,brisc2025_train_03705_pi_ax_t1.png,brisc2025_test_00737_pi_ax_t1.png,True,False,1452,OK
3,brisc2025_train_04176_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,brisc2025_train_04176_pi_co_t1.png,brisc2025_test_00825_pi_co_t1.png,True,False,2148,OK
4,brisc2025_train_04177_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,brisc2025_train_04177_pi_co_t1.png,brisc2025_test_00825_pi_co_t1.png,True,False,2148,OK
5,brisc2025_train_04178_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,brisc2025_train_04178_pi_co_t1.png,brisc2025_test_00825_pi_co_t1.png,True,False,2148,OK
6,brisc2025_train_01210_me_ax_t1.jpg,brisc2025_test_00344_me_ax_t1.jpg,brisc2025_train_01210_me_ax_t1.png,brisc2025_test_00344_me_ax_t1.png,True,False,1538,OK
7,brisc2025_train_04218_pi_co_t1.jpg,brisc2025_test_00834_pi_co_t1.jpg,brisc2025_train_04218_pi_co_t1.png,brisc2025_test_00834_pi_co_t1.png,True,False,2567,OK
8,brisc2025_train_01304_me_ax_t1.jpg,brisc2025_test_00352_me_ax_t1.jpg,brisc2025_train_01304_me_ax_t1.png,brisc2025_test_00352_me_ax_t1.png,True,False,567,OK



SUMMARY
--------------------------------------------------
Comparisons attempted: 9
Mask links successfully found: 9
Pixel-identical mask pairs: 0
Pixel-different mask pairs: 9


### 8.4 Quantitative Comparison of Duplicate-MRI Masks

In [24]:
def mask_similarity_metrics(mask1_path, mask2_path):
    with Image.open(mask1_path) as im1:
        arr1 = np.array(im1)

    with Image.open(mask2_path) as im2:
        arr2 = np.array(im2)

    # Convert to binary masks
    bin1 = arr1 > 0
    bin2 = arr2 > 0

    intersection = np.logical_and(bin1, bin2).sum()
    union = np.logical_or(bin1, bin2).sum()

    area1 = bin1.sum()
    area2 = bin2.sum()

    dice = (
        2 * intersection / (area1 + area2)
        if (area1 + area2) > 0
        else 1.0
    )

    iou = (
        intersection / union
        if union > 0
        else 1.0
    )

    return {
        "train_mask_area": int(area1),
        "test_mask_area": int(area2),
        "intersection": int(intersection),
        "union": int(union),
        "dice": float(dice),
        "iou": float(iou)
    }


metric_results = []

for _, row in mask_comparison_df.iterrows():

    train_mask_row = mask_df[
        mask_df["filename"] == row["train_mask"]
    ].iloc[0]

    test_mask_row = mask_df[
        mask_df["filename"] == row["test_mask"]
    ].iloc[0]

    train_mask_path = (
        DATASET_ROOT /
        Path(train_mask_row["relative_path"])
    )

    test_mask_path = (
        DATASET_ROOT /
        Path(test_mask_row["relative_path"])
    )

    metrics = mask_similarity_metrics(
        train_mask_path,
        test_mask_path
    )

    metric_results.append({
        "train_image": row["train_image"],
        "test_image": row["test_image"],
        "different_pixels": row["different_pixels"],
        **metrics
    })


mask_similarity_df = pd.DataFrame(metric_results)

display(mask_similarity_df)

print("\nSUMMARY")
print("-" * 50)

print(
    "Mean Dice:",
    round(mask_similarity_df["dice"].mean(), 4)
)

print(
    "Minimum Dice:",
    round(mask_similarity_df["dice"].min(), 4)
)

print(
    "Maximum Dice:",
    round(mask_similarity_df["dice"].max(), 4)
)

print(
    "Mean IoU:",
    round(mask_similarity_df["iou"].mean(), 4)
)

print(
    "Minimum IoU:",
    round(mask_similarity_df["iou"].min(), 4)
)

print(
    "Maximum IoU:",
    round(mask_similarity_df["iou"].max(), 4)
)

,train_image,test_image,different_pixels,train_mask_area,test_mask_area,intersection,union,dice,iou
0,brisc2025_train_04194_pi_co_t1.jpg,brisc2025_test_00827_pi_co_t1.jpg,789,1628,1612,1420,1820,0.876543,0.780220
1,brisc2025_train_03749_pi_ax_t1.jpg,brisc2025_test_00751_pi_ax_t1.jpg,1475,4102,4052,3678,4476,0.902134,0.821716
2,brisc2025_train_03705_pi_ax_t1.jpg,brisc2025_test_00737_pi_ax_t1.jpg,1452,2895,2771,2425,3241,0.855983,0.748226
3,brisc2025_train_04176_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,2148,4809,4045,3702,5152,0.836232,0.718556
4,brisc2025_train_04177_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,2148,4809,4045,3702,5152,0.836232,0.718556
5,brisc2025_train_04178_pi_co_t1.jpg,brisc2025_test_00825_pi_co_t1.jpg,2148,4809,4045,3702,5152,0.836232,0.718556
6,brisc2025_train_01210_me_ax_t1.jpg,brisc2025_test_00344_me_ax_t1.jpg,1538,5081,5488,4898,5671,0.926862,0.863692
7,brisc2025_train_04218_pi_co_t1.jpg,brisc2025_test_00834_pi_co_t1.jpg,2567,5226,5630,4837,6019,0.891120,0.803622
8,brisc2025_train_01304_me_ax_t1.jpg,brisc2025_test_00352_me_ax_t1.jpg,567,760,838,657,941,0.822278,0.698193



SUMMARY
--------------------------------------------------
Mean Dice: 0.8648
Minimum Dice: 0.8223
Maximum Dice: 0.9269
Mean IoU: 0.7635
Minimum IoU: 0.6982
Maximum IoU: 0.8637


### Conclusion of Exact Train–Test Duplicate Investigation

The BRISC integrity audit identified seven unique MRI SHA-256 hashes that
occur in both the official training and testing splits. These correspond to
nine training records and seven testing records.

The same seven cross-split MRI hashes occur in both the classification and
segmentation task structures.

For classification, the duplicated images retain consistent tumor labels,
but their presence in both training and testing constitutes image-level
train–test leakage.

For segmentation, the duplicated MRI inputs are byte-identical, while their
corresponding masks are not pixel-identical. Across nine train-to-test mask
comparisons, the mean Dice coefficient was 0.8648 (range 0.8223–0.9269) and
the mean IoU was 0.7635 (range 0.6982–0.8637). Thus, the annotations show
substantial spatial overlap but meaningful differences.

The original BRISC files will remain unchanged. A reproducible
leakage-controlled experimental protocol will exclude training records that
duplicate official test inputs before model development.

## 9. Near-Duplicate Train–Test Detection

Exact SHA-256 matching detects only byte-identical files. This section uses
perceptual hashing to identify possible visually near-identical MRI images
across the official classification training and testing splits.

Perceptual-hash matches are treated only as candidates and require secondary
pixel-level/visual verification before being classified as near-duplicates.

In [25]:
try:
    import imagehash
    print("ImageHash is already installed.")
except ImportError:
    print("ImageHash is not installed.")

ImageHash is not installed.


In [26]:
%pip install ImageHash

   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------------------- -------------------- 2.1/4.2 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------  4.2/4.2 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 9.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [PyWavelets]
   ---------------------------------------- 2/2 [ImageHash]

Note: you may need to restart the kernel to use updated packages.


In [27]:
import imagehash
print("ImageHash imported successfully.")

ImageHash imported successfully.


### 9.2 Computing Perceptual Hashes

In [28]:
from PIL import Image
import imagehash
import pandas as pd


def compute_phash(relative_path):
    image_path = DATASET_ROOT / Path(relative_path)

    with Image.open(image_path) as img:
        # Convert consistently to grayscale before hashing
        img = img.convert("L")
        return str(imagehash.phash(img, hash_size=8))


classification_phash_df = classification_df[
    [
        "filename",
        "split",
        "tumor_label",
        "plane_label",
        "sequence",
        "width",
        "height",
        "sha256",
        "relative_path"
    ]
].copy()


print("Computing perceptual hashes...")

classification_phash_df["phash"] = (
    classification_phash_df["relative_path"]
    .apply(compute_phash)
)

print("Done.")
print("Images hashed:", len(classification_phash_df))

display(classification_phash_df.head())

Computing perceptual hashes...
Done.
Images hashed: 6000


,filename,split,tumor_label,plane_label,sequence,width,height,sha256,relative_path,phash
0,brisc2025_test_00001_gl_ax_t1.jpg,test,glioma,axial,T1,512,512,b513a83f68e486f45159b004d356782e78284014532151...,classification_task\test\glioma\brisc2025_test...,93c62f3a6c6824cf
1,brisc2025_test_00002_gl_ax_t1.jpg,test,glioma,axial,T1,512,512,5b4e4a0f482cb68955c340205d475f25c3a30718bd3b48...,classification_task\test\glioma\brisc2025_test...,c2632b1c3c3c675b
2,brisc2025_test_00003_gl_ax_t1.jpg,test,glioma,axial,T1,512,512,6fd1918b39e7a7f92fa5652d7802583f3251fed50a23ec...,classification_task\test\glioma\brisc2025_test...,c23a3de170cf638c
3,brisc2025_test_00004_gl_ax_t1.jpg,test,glioma,axial,T1,512,512,216b65a45d7effb4ebce1c9893f97d12449c9da777f818...,classification_task\test\glioma\brisc2025_test...,c3923f69648333cd
4,brisc2025_test_00005_gl_ax_t1.jpg,test,glioma,axial,T1,512,512,c35734545021ab5a4f55fb87e08732e26fb16abe887927...,classification_task\test\glioma\brisc2025_test...,c2962f7934c730cd


### 9.3 Cross-Split Perceptual-Hash Candidate Search

In [29]:
train_phash_df = classification_phash_df[
    classification_phash_df["split"] == "train"
].copy()

test_phash_df = classification_phash_df[
    classification_phash_df["split"] == "test"
].copy()


# Convert 64-bit hexadecimal pHash to integers
train_phash_df["phash_int"] = train_phash_df["phash"].apply(
    lambda x: int(x, 16)
)

test_phash_df["phash_int"] = test_phash_df["phash"].apply(
    lambda x: int(x, 16)
)


near_duplicate_candidates = []

# Strict initial threshold
PHASH_THRESHOLD = 4


for _, test_row in test_phash_df.iterrows():

    # Start conservatively:
    # compare only against training images with the same tumor label.
    candidate_train = train_phash_df[
        train_phash_df["tumor_label"] == test_row["tumor_label"]
    ]

    for _, train_row in candidate_train.iterrows():

        # Skip already-known byte-identical files.
        # We are looking for additional near-duplicates here.
        if train_row["sha256"] == test_row["sha256"]:
            continue

        distance = (
            int(train_row["phash_int"]) ^
            int(test_row["phash_int"])
        ).bit_count()

        if distance <= PHASH_THRESHOLD:

            near_duplicate_candidates.append({
                "train_filename": train_row["filename"],
                "test_filename": test_row["filename"],
                "tumor_label": test_row["tumor_label"],
                "train_plane": train_row["plane_label"],
                "test_plane": test_row["plane_label"],
                "train_size": (
                    int(train_row["width"]),
                    int(train_row["height"])
                ),
                "test_size": (
                    int(test_row["width"]),
                    int(test_row["height"])
                ),
                "phash_distance": distance,
                "train_sha256": train_row["sha256"],
                "test_sha256": test_row["sha256"],
                "train_relative_path": train_row["relative_path"],
                "test_relative_path": test_row["relative_path"]
            })


near_duplicate_candidates_df = pd.DataFrame(
    near_duplicate_candidates
)

print("pHash threshold:", PHASH_THRESHOLD)
print(
    "Candidate train/test pairs:",
    len(near_duplicate_candidates_df)
)

if not near_duplicate_candidates_df.empty:

    print("\nCandidates by pHash distance:")
    print(
        near_duplicate_candidates_df[
            "phash_distance"
        ].value_counts().sort_index()
    )

    display(
        near_duplicate_candidates_df
        .sort_values("phash_distance")
        .head(30)
    )
else:
    print(
        "\nNo additional near-duplicate candidates "
        "found at this threshold."
    )

pHash threshold: 4
Candidate train/test pairs: 428

Candidates by pHash distance:
phash_distance
0    119
2     85
4    224
Name: count, dtype: int64


,train_filename,test_filename,tumor_label,train_plane,test_plane,train_size,test_size,phash_distance,train_sha256,test_sha256,train_relative_path,test_relative_path
213,brisc2025_train_01667_me_co_t1.jpg,brisc2025_test_00415_me_co_t1.jpg,meningioma,coronal,coronal,"(341, 395)","(341, 395)",0,e2c7bc9c2d735e5b584b329f5cda1aec1f2415b2972c4a...,f55311fb8ccf1b86450233262bd00b176bd1c404e5849b...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
104,brisc2025_train_01291_me_ax_t1.jpg,brisc2025_test_00276_me_ax_t1.jpg,meningioma,axial,axial,"(225, 225)","(442, 442)",0,e465760e8b0ca460951cfa70174020194061c32793bc23...,b31d723ab6491c8da12fddc8b756b398e5f35cf08f063f...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
207,brisc2025_train_01661_me_co_t1.jpg,brisc2025_test_00404_me_co_t1.jpg,meningioma,coronal,coronal,"(313, 393)","(313, 393)",0,7efcd87636c28a674f2abf3bc6255eb1e786c12f40e794...,066502884d443971d71836da239f41751eefdbb164af95...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
106,brisc2025_train_01271_me_ax_t1.jpg,brisc2025_test_00278_me_ax_t1.jpg,meningioma,axial,axial,"(312, 401)","(312, 401)",0,7ae82ad05ee2fd7dc4d6ee5f4793ac4fc8c416bba68b69...,4b341616b6b9a8c0060988aa8174ee101aca4f96ae8007...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
107,brisc2025_train_01272_me_ax_t1.jpg,brisc2025_test_00279_me_ax_t1.jpg,meningioma,axial,axial,"(300, 412)","(300, 412)",0,3f485834e521c740f298d16190f0afa1209a0e2d010800...,4cc66638107c582da061fb013badc87dfb5dbdb94deee5...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
108,brisc2025_train_01273_me_ax_t1.jpg,brisc2025_test_00280_me_ax_t1.jpg,meningioma,axial,axial,"(205, 251)","(205, 251)",0,df42fa96f4e750ec67b95889f912770e0408328899912b...,87886ae7a7ec7535a975fa0b267d5889a995699a4fd5d7...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
109,brisc2025_train_01274_me_ax_t1.jpg,brisc2025_test_00282_me_ax_t1.jpg,meningioma,axial,axial,"(204, 249)","(204, 249)",0,7cfaba22355b770b309b9e019fb8a86ad4e3f22f16c00b...,cdc4d724a668388abc3657213ea91c3e8d849b2c7e155d...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
110,brisc2025_train_01275_me_ax_t1.jpg,brisc2025_test_00283_me_ax_t1.jpg,meningioma,axial,axial,"(323, 342)","(323, 342)",0,c4eae93e7fb065a8f96e0c916bced76f23e218331d5c54...,935ba7f0696adb99b0bc47bd75e6ae0ae86426d1fb6922...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
103,brisc2025_train_01269_me_ax_t1.jpg,brisc2025_test_00276_me_ax_t1.jpg,meningioma,axial,axial,"(442, 442)","(442, 442)",0,f260fc621e5ae555a5eaaab2dcb29d491386a3849191b0...,b31d723ab6491c8da12fddc8b756b398e5f35cf08f063f...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...
111,brisc2025_train_01255_me_ax_t1.jpg,brisc2025_test_00284_me_ax_t1.jpg,meningioma,axial,axial,"(369, 398)","(369, 398)",0,d3dc17e9beb57b30ad2dee3323cb2a7a7e893bcc0666b7...,ca02d2e726540598237ff366f8a9d4078f5f045bc85b2a...,classification_task\train\meningioma\brisc2025...,classification_task\test\meningioma\brisc2025_...


### 9.4 Save Perceptual-Hash Candidate List

In [31]:
near_duplicate_candidates_df.to_csv(
    INTERIM_ROOT / "classification_phash_candidates.csv",
    index=False
)

print("Saved candidate pairs:",
      len(near_duplicate_candidates_df))

print(
    "Saved to:",
    INTERIM_ROOT / "classification_phash_candidates.csv"
)

Saved candidate pairs: 428
Saved to: C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\interim\classification_phash_candidates.csv


### 9.5 Pixel-Level Verification of pHash Candidates

In [32]:
import numpy as np
from PIL import Image


def load_normalized_image(relative_path, size=(256, 256)):
    image_path = DATASET_ROOT / Path(relative_path)

    with Image.open(image_path) as img:
        img = img.convert("L")
        img = img.resize(size, Image.Resampling.BILINEAR)

        arr = np.asarray(img, dtype=np.float32) / 255.0

    return arr


verification_results = []


for _, row in near_duplicate_candidates_df.iterrows():

    train_arr = load_normalized_image(
        row["train_relative_path"]
    )

    test_arr = load_normalized_image(
        row["test_relative_path"]
    )

    # Root Mean Squared Error
    rmse = np.sqrt(
        np.mean((train_arr - test_arr) ** 2)
    )

    # Pearson pixel correlation
    train_flat = train_arr.ravel()
    test_flat = test_arr.ravel()

    if (
        np.std(train_flat) > 0 and
        np.std(test_flat) > 0
    ):
        correlation = np.corrcoef(
            train_flat,
            test_flat
        )[0, 1]
    else:
        correlation = np.nan


    verification_results.append({
        "train_filename": row["train_filename"],
        "test_filename": row["test_filename"],
        "tumor_label": row["tumor_label"],
        "train_plane": row["train_plane"],
        "test_plane": row["test_plane"],
        "train_size": row["train_size"],
        "test_size": row["test_size"],
        "phash_distance": row["phash_distance"],
        "rmse": float(rmse),
        "correlation": float(correlation),
        "train_relative_path": row[
            "train_relative_path"
        ],
        "test_relative_path": row[
            "test_relative_path"
        ]
    })


near_duplicate_verification_df = pd.DataFrame(
    verification_results
)

print(
    "Candidate pairs verified:",
    len(near_duplicate_verification_df)
)

Candidate pairs verified: 428


In [33]:
print("PIXEL-SIMILARITY SUMMARY")
print("-" * 50)

print(
    "RMSE minimum:",
    round(
        near_duplicate_verification_df["rmse"].min(),
        6
    )
)

print(
    "RMSE median:",
    round(
        near_duplicate_verification_df["rmse"].median(),
        6
    )
)

print(
    "RMSE maximum:",
    round(
        near_duplicate_verification_df["rmse"].max(),
        6
    )
)

print()

print(
    "Correlation minimum:",
    round(
        near_duplicate_verification_df[
            "correlation"
        ].min(),
        6
    )
)

print(
    "Correlation median:",
    round(
        near_duplicate_verification_df[
            "correlation"
        ].median(),
        6
    )
)

print(
    "Correlation maximum:",
    round(
        near_duplicate_verification_df[
            "correlation"
        ].max(),
        6
    )
)

PIXEL-SIMILARITY SUMMARY
--------------------------------------------------
RMSE minimum: 0.0
RMSE median: 0.070302
RMSE maximum: 0.189997

Correlation minimum: 0.579316
Correlation median: 0.902845
Correlation maximum: 1.0


In [34]:
strong_candidates = (
    near_duplicate_verification_df[
        (near_duplicate_verification_df["correlation"] >= 0.995) &
        (near_duplicate_verification_df["rmse"] <= 0.03)
    ]
    .sort_values(
        ["correlation", "rmse"],
        ascending=[False, True]
    )
)

print(
    "Strong near-duplicate candidates:",
    len(strong_candidates)
)

display(
    strong_candidates[
        [
            "train_filename",
            "test_filename",
            "tumor_label",
            "train_plane",
            "test_plane",
            "train_size",
            "test_size",
            "phash_distance",
            "rmse",
            "correlation"
        ]
    ].head(30)
)

Strong near-duplicate candidates: 122


,train_filename,test_filename,tumor_label,train_plane,test_plane,train_size,test_size,phash_distance,rmse,correlation
70,brisc2025_train_01261_me_ax_t1.jpg,brisc2025_test_00255_me_ax_t1.jpg,meningioma,axial,axial,"(257, 307)","(257, 307)",0,0.0,1.0
73,brisc2025_train_01402_me_ax_t1.jpg,brisc2025_test_00257_me_ax_t1.jpg,meningioma,axial,axial,"(528, 581)","(528, 581)",0,0.0,1.0
75,brisc2025_train_01381_me_ax_t1.jpg,brisc2025_test_00259_me_ax_t1.jpg,meningioma,axial,axial,"(219, 234)","(219, 234)",0,0.0,1.0
76,brisc2025_train_01383_me_ax_t1.jpg,brisc2025_test_00260_me_ax_t1.jpg,meningioma,axial,axial,"(206, 249)","(206, 249)",0,0.0,1.0
78,brisc2025_train_01384_me_ax_t1.jpg,brisc2025_test_00261_me_ax_t1.jpg,meningioma,axial,axial,"(530, 526)","(530, 526)",0,0.0,1.0
82,brisc2025_train_01262_me_ax_t1.jpg,brisc2025_test_00263_me_ax_t1.jpg,meningioma,axial,axial,"(200, 223)","(200, 223)",0,0.0,1.0
93,brisc2025_train_01265_me_ax_t1.jpg,brisc2025_test_00271_me_ax_t1.jpg,meningioma,axial,axial,"(318, 354)","(318, 354)",0,0.0,1.0
98,brisc2025_train_01266_me_ax_t1.jpg,brisc2025_test_00273_me_ax_t1.jpg,meningioma,axial,axial,"(315, 341)","(315, 341)",0,0.0,1.0
99,brisc2025_train_01267_me_ax_t1.jpg,brisc2025_test_00274_me_ax_t1.jpg,meningioma,axial,axial,"(278, 306)","(278, 306)",0,0.0,1.0
108,brisc2025_train_01273_me_ax_t1.jpg,brisc2025_test_00280_me_ax_t1.jpg,meningioma,axial,axial,"(205, 251)","(205, 251)",0,0.0,1.0


### 9.6 Characterization of Strong Near-Duplicate Candidates

In [35]:
print("STRONG-CANDIDATE SUMMARY")
print("-" * 55)

print("Total strong candidate pairs:", len(strong_candidates))

print(
    "Unique training images involved:",
    strong_candidates["train_filename"].nunique()
)

print(
    "Unique testing images involved:",
    strong_candidates["test_filename"].nunique()
)

print("\nPairs by tumor class:")
print(
    strong_candidates["tumor_label"]
    .value_counts()
)

print("\nPairs by pHash distance:")
print(
    strong_candidates["phash_distance"]
    .value_counts()
    .sort_index()
)


# Pairs that became completely identical on the
# normalized 256x256 grayscale comparison grid
perfect_normalized_matches = strong_candidates[
    np.isclose(strong_candidates["rmse"], 0.0, atol=1e-12) &
    np.isclose(strong_candidates["correlation"], 1.0, atol=1e-12)
].copy()


print("\nPERFECT NORMALIZED MATCHES")
print("-" * 55)

print(
    "Pairs with RMSE=0 and correlation=1:",
    len(perfect_normalized_matches)
)

print(
    "Unique training images:",
    perfect_normalized_matches["train_filename"].nunique()
)

print(
    "Unique testing images:",
    perfect_normalized_matches["test_filename"].nunique()
)

print("\nBy tumor class:")
print(
    perfect_normalized_matches["tumor_label"]
    .value_counts()
)

STRONG-CANDIDATE SUMMARY
-------------------------------------------------------
Total strong candidate pairs: 122
Unique training images involved: 98
Unique testing images involved: 96

Pairs by tumor class:
tumor_label
meningioma    122
Name: count, dtype: int64

Pairs by pHash distance:
phash_distance
0    114
2      8
Name: count, dtype: int64

PERFECT NORMALIZED MATCHES
-------------------------------------------------------
Pairs with RMSE=0 and correlation=1: 101
Unique training images: 96
Unique testing images: 94

By tumor class:
tumor_label
meningioma    101
Name: count, dtype: int64


### 9.7 Native-Resolution Verification of Perfect Normalized Matches

In [36]:
native_results = []

for _, row in perfect_normalized_matches.iterrows():

    train_path = (
        DATASET_ROOT /
        Path(row["train_relative_path"])
    )

    test_path = (
        DATASET_ROOT /
        Path(row["test_relative_path"])
    )

    with Image.open(train_path) as img1:
        arr1 = np.asarray(
            img1.convert("L"),
            dtype=np.uint8
        )

    with Image.open(test_path) as img2:
        arr2 = np.asarray(
            img2.convert("L"),
            dtype=np.uint8
        )

    same_native_shape = arr1.shape == arr2.shape

    if same_native_shape:

        native_pixel_identical = np.array_equal(
            arr1,
            arr2
        )

        native_rmse = (
            np.sqrt(
                np.mean(
                    (
                        arr1.astype(np.float32) -
                        arr2.astype(np.float32)
                    ) ** 2
                )
            )
            / 255.0
        )

    else:
        native_pixel_identical = False
        native_rmse = None

    native_results.append({
        "train_filename": row["train_filename"],
        "test_filename": row["test_filename"],
        "tumor_label": row["tumor_label"],
        "train_size": row["train_size"],
        "test_size": row["test_size"],
        "same_native_shape": same_native_shape,
        "native_pixel_identical": native_pixel_identical,
        "native_rmse": native_rmse
    })


native_verification_df = pd.DataFrame(native_results)


print("NATIVE-RESOLUTION VERIFICATION")
print("-" * 55)

print(
    "Pairs checked:",
    len(native_verification_df)
)

print(
    "Same native dimensions:",
    native_verification_df["same_native_shape"].sum()
)

print(
    "Different native dimensions:",
    (~native_verification_df["same_native_shape"]).sum()
)

print(
    "Pixel-identical at native resolution:",
    native_verification_df[
        "native_pixel_identical"
    ].sum()
)


same_shape_nonidentical = native_verification_df[
    (native_verification_df["same_native_shape"] == True) &
    (native_verification_df["native_pixel_identical"] == False)
]

print(
    "Same dimensions but non-identical pixels:",
    len(same_shape_nonidentical)
)


display(
    native_verification_df.head(30)
)

NATIVE-RESOLUTION VERIFICATION
-------------------------------------------------------
Pairs checked: 101
Same native dimensions: 101
Different native dimensions: 0
Pixel-identical at native resolution: 101
Same dimensions but non-identical pixels: 0


,train_filename,test_filename,tumor_label,train_size,test_size,same_native_shape,native_pixel_identical,native_rmse
0,brisc2025_train_01261_me_ax_t1.jpg,brisc2025_test_00255_me_ax_t1.jpg,meningioma,"(257, 307)","(257, 307)",True,True,0.0
1,brisc2025_train_01402_me_ax_t1.jpg,brisc2025_test_00257_me_ax_t1.jpg,meningioma,"(528, 581)","(528, 581)",True,True,0.0
2,brisc2025_train_01381_me_ax_t1.jpg,brisc2025_test_00259_me_ax_t1.jpg,meningioma,"(219, 234)","(219, 234)",True,True,0.0
3,brisc2025_train_01383_me_ax_t1.jpg,brisc2025_test_00260_me_ax_t1.jpg,meningioma,"(206, 249)","(206, 249)",True,True,0.0
4,brisc2025_train_01384_me_ax_t1.jpg,brisc2025_test_00261_me_ax_t1.jpg,meningioma,"(530, 526)","(530, 526)",True,True,0.0
5,brisc2025_train_01262_me_ax_t1.jpg,brisc2025_test_00263_me_ax_t1.jpg,meningioma,"(200, 223)","(200, 223)",True,True,0.0
6,brisc2025_train_01265_me_ax_t1.jpg,brisc2025_test_00271_me_ax_t1.jpg,meningioma,"(318, 354)","(318, 354)",True,True,0.0
7,brisc2025_train_01266_me_ax_t1.jpg,brisc2025_test_00273_me_ax_t1.jpg,meningioma,"(315, 341)","(315, 341)",True,True,0.0
8,brisc2025_train_01267_me_ax_t1.jpg,brisc2025_test_00274_me_ax_t1.jpg,meningioma,"(278, 306)","(278, 306)",True,True,0.0
9,brisc2025_train_01273_me_ax_t1.jpg,brisc2025_test_00280_me_ax_t1.jpg,meningioma,"(205, 251)","(205, 251)",True,True,0.0


In [37]:
native_identical_df = native_verification_df[
    native_verification_df["native_pixel_identical"] == True
]

different_resolution_df = native_verification_df[
    native_verification_df["same_native_shape"] == False
]


print("HIGH-CONFIDENCE GROUP SUMMARY")
print("-" * 55)

print("\nNative pixel-identical pairs:")
print(len(native_identical_df))

print(
    "Unique training images:",
    native_identical_df["train_filename"].nunique()
)

print(
    "Unique testing images:",
    native_identical_df["test_filename"].nunique()
)


print("\nPerfect-normalized pairs with different native resolutions:")
print(len(different_resolution_df))

print(
    "Unique training images:",
    different_resolution_df["train_filename"].nunique()
)

print(
    "Unique testing images:",
    different_resolution_df["test_filename"].nunique()
)

HIGH-CONFIDENCE GROUP SUMMARY
-------------------------------------------------------

Native pixel-identical pairs:
101
Unique training images: 96
Unique testing images: 94

Perfect-normalized pairs with different native resolutions:
0
Unique training images: 0
Unique testing images: 0


### 9.8 Full RGB Pixel Verification of Native Duplicate Candidates

In [38]:
rgb_results = []

for _, row in perfect_normalized_matches.iterrows():

    train_path = (
        DATASET_ROOT /
        Path(row["train_relative_path"])
    )

    test_path = (
        DATASET_ROOT /
        Path(row["test_relative_path"])
    )

    with Image.open(train_path) as img1:
        train_original_mode = img1.mode
        arr1 = np.asarray(
            img1.convert("RGB"),
            dtype=np.uint8
        )

    with Image.open(test_path) as img2:
        test_original_mode = img2.mode
        arr2 = np.asarray(
            img2.convert("RGB"),
            dtype=np.uint8
        )

    same_shape = arr1.shape == arr2.shape

    rgb_identical = (
        same_shape and
        np.array_equal(arr1, arr2)
    )

    if same_shape:
        different_rgb_values = int(
            np.count_nonzero(arr1 != arr2)
        )
    else:
        different_rgb_values = None

    rgb_results.append({
        "train_filename": row["train_filename"],
        "test_filename": row["test_filename"],
        "tumor_label": row["tumor_label"],
        "train_original_mode": train_original_mode,
        "test_original_mode": test_original_mode,
        "same_rgb_shape": same_shape,
        "rgb_pixel_identical": rgb_identical,
        "different_rgb_values": different_rgb_values
    })


rgb_verification_df = pd.DataFrame(rgb_results)

print("FULL RGB VERIFICATION")
print("-" * 55)

print("Pairs checked:", len(rgb_verification_df))

print(
    "RGB pixel-identical pairs:",
    rgb_verification_df[
        "rgb_pixel_identical"
    ].sum()
)

print(
    "RGB non-identical pairs:",
    (~rgb_verification_df[
        "rgb_pixel_identical"
    ]).sum()
)

print("\nOriginal mode combinations:")
print(
    rgb_verification_df[
        ["train_original_mode", "test_original_mode"]
    ].value_counts()
)

display(rgb_verification_df.head(20))

FULL RGB VERIFICATION
-------------------------------------------------------
Pairs checked: 101
RGB pixel-identical pairs: 101
RGB non-identical pairs: 0

Original mode combinations:
train_original_mode  test_original_mode
RGB                  RGB                   101
Name: count, dtype: int64


,train_filename,test_filename,tumor_label,train_original_mode,test_original_mode,same_rgb_shape,rgb_pixel_identical,different_rgb_values
0,brisc2025_train_01261_me_ax_t1.jpg,brisc2025_test_00255_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
1,brisc2025_train_01402_me_ax_t1.jpg,brisc2025_test_00257_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
2,brisc2025_train_01381_me_ax_t1.jpg,brisc2025_test_00259_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
3,brisc2025_train_01383_me_ax_t1.jpg,brisc2025_test_00260_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
4,brisc2025_train_01384_me_ax_t1.jpg,brisc2025_test_00261_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
5,brisc2025_train_01262_me_ax_t1.jpg,brisc2025_test_00263_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
6,brisc2025_train_01265_me_ax_t1.jpg,brisc2025_test_00271_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
7,brisc2025_train_01266_me_ax_t1.jpg,brisc2025_test_00273_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
8,brisc2025_train_01267_me_ax_t1.jpg,brisc2025_test_00274_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0
9,brisc2025_train_01273_me_ax_t1.jpg,brisc2025_test_00280_me_ax_t1.jpg,meningioma,RGB,RGB,True,True,0


### 9.9 Consolidation of High-Confidence Cross-Split Duplicates

This section combines file-level SHA-256 duplicates and decoded RGB
pixel-identical duplicates to determine the final number of unique training
and testing images affected by confirmed cross-split duplication.

In [39]:
# --------------------------------------------------
# A. Test/train images involved in exact SHA overlap
# --------------------------------------------------

sha_overlap_train_images = set(
    cross_split_duplicates.loc[
        cross_split_duplicates["split"] == "train",
        "filename"
    ]
)

sha_overlap_test_images = set(
    cross_split_duplicates.loc[
        cross_split_duplicates["split"] == "test",
        "filename"
    ]
)


# --------------------------------------------------
# B. Test/train images involved in RGB pixel-identical
#    but different-SHA overlap
# --------------------------------------------------

rgb_duplicate_train_images = set(
    rgb_verification_df.loc[
        rgb_verification_df["rgb_pixel_identical"] == True,
        "train_filename"
    ]
)

rgb_duplicate_test_images = set(
    rgb_verification_df.loc[
        rgb_verification_df["rgb_pixel_identical"] == True,
        "test_filename"
    ]
)


# --------------------------------------------------
# C. Intersections
# --------------------------------------------------

train_intersection = (
    sha_overlap_train_images &
    rgb_duplicate_train_images
)

test_intersection = (
    sha_overlap_test_images &
    rgb_duplicate_test_images
)


# --------------------------------------------------
# D. Final unions
# --------------------------------------------------

confirmed_train_images = (
    sha_overlap_train_images |
    rgb_duplicate_train_images
)

confirmed_test_images = (
    sha_overlap_test_images |
    rgb_duplicate_test_images
)


print("CONFIRMED CROSS-SPLIT DUPLICATION SUMMARY")
print("-" * 60)

print("\nSHA-256 exact-overlap:")
print("Unique training images:", len(sha_overlap_train_images))
print("Unique testing images :", len(sha_overlap_test_images))

print("\nRGB pixel-identical / different-SHA:")
print("Unique training images:", len(rgb_duplicate_train_images))
print("Unique testing images :", len(rgb_duplicate_test_images))

print("\nOverlap between the two detection groups:")
print("Training images in both groups:", len(train_intersection))
print("Testing images in both groups :", len(test_intersection))

print("\nFINAL UNIQUE CONFIRMED SET:")
print("Unique training images affected:", len(confirmed_train_images))
print("Unique testing images affected :", len(confirmed_test_images))

CONFIRMED CROSS-SPLIT DUPLICATION SUMMARY
------------------------------------------------------------

SHA-256 exact-overlap:
Unique training images: 9
Unique testing images : 7

RGB pixel-identical / different-SHA:
Unique training images: 96
Unique testing images : 94

Overlap between the two detection groups:
Training images in both groups: 1
Testing images in both groups : 0

FINAL UNIQUE CONFIRMED SET:
Unique training images affected: 104
Unique testing images affected : 101


In [40]:
confirmed_test_df = classification_df[
    classification_df["filename"].isin(confirmed_test_images)
].copy()

confirmed_train_df = classification_df[
    classification_df["filename"].isin(confirmed_train_images)
].copy()


print("CONFIRMED TEST IMAGES BY CLASS")
print("-" * 50)

test_by_class = (
    confirmed_test_df["tumor_label"]
    .value_counts()
    .sort_index()
)

print(test_by_class)


print("\nTEST-SET CONTAMINATION RATE BY CLASS")
print("-" * 50)

all_test_counts = (
    classification_df[
        classification_df["split"] == "test"
    ]["tumor_label"]
    .value_counts()
    .sort_index()
)

for tumor_class in all_test_counts.index:

    total = int(all_test_counts[tumor_class])

    affected = int(
        test_by_class.get(tumor_class, 0)
    )

    percentage = 100 * affected / total

    print(
        f"{tumor_class:<12}: "
        f"{affected:>3}/{total:<3} "
        f"({percentage:.2f}%)"
    )


overall_affected = len(confirmed_test_images)
overall_test = 1000

print(
    f"\nOverall confirmed test contamination: "
    f"{overall_affected}/{overall_test} "
    f"({100 * overall_affected / overall_test:.2f}%)"
)

CONFIRMED TEST IMAGES BY CLASS
--------------------------------------------------
tumor_label
meningioma    96
pituitary      5
Name: count, dtype: int64

TEST-SET CONTAMINATION RATE BY CLASS
--------------------------------------------------
glioma      :   0/254 (0.00%)
meningioma  :  96/306 (31.37%)
no_tumor    :   0/140 (0.00%)
pituitary   :   5/300 (1.67%)

Overall confirmed test contamination: 101/1000 (10.10%)


In [41]:
rgb_verification_df.to_csv(
    INTERIM_ROOT /
    "classification_rgb_pixel_duplicate_pairs.csv",
    index=False
)

confirmed_test_df.to_csv(
    INTERIM_ROOT /
    "classification_confirmed_leaked_test_images.csv",
    index=False
)

confirmed_train_df.to_csv(
    INTERIM_ROOT /
    "classification_confirmed_duplicate_train_images.csv",
    index=False
)

print("Confirmed duplicate audit files saved.")

Confirmed duplicate audit files saved.


### Confirmed Decoded-Content Duplicate Finding

Perceptual-hash screening initially identified possible cross-split
near-duplicates. Secondary verification showed that 101 train-test pairs
have identical decoded RGB pixel arrays despite having different file-level
SHA-256 hashes.

All 101 pairs have identical native dimensions and zero differing RGB pixel
values. Therefore, these records are treated as decoded-content exact
duplicates rather than merely visually similar images.

These duplicate pairs are concentrated in the meningioma class. They are
combined with the independently detected SHA-256 exact-overlap records to
construct a final unique set of confirmed cross-split duplicate images.
The raw BRISC dataset remains unchanged.

## 10. Final Integrity Audit Outputs

The raw BRISC dataset is preserved without modification.

Confirmed train–test duplicates are recorded separately so that subsequent
experiments can construct a leakage-controlled training set while preserving
the original dataset for reproducibility.

Only confirmed duplicates are included in the automatic exclusion list.
High-similarity but non-identical candidates remain documented separately and
are not automatically excluded.

In [42]:
# --------------------------------------------------
# 1. Confirmed training exclusion list
# --------------------------------------------------

train_exclusion_df = classification_df[
    classification_df["filename"].isin(confirmed_train_images)
].copy()

train_exclusion_df["exclusion_reason"] = (
    "confirmed_cross_split_duplicate"
)

train_exclusion_df.to_csv(
    INTERIM_ROOT /
    "classification_train_exclusion_confirmed_duplicates.csv",
    index=False
)


# --------------------------------------------------
# 2. Confirmed contaminated test references
#    DO NOT remove these automatically.
# --------------------------------------------------

test_leakage_reference_df = classification_df[
    classification_df["filename"].isin(confirmed_test_images)
].copy()

test_leakage_reference_df["audit_status"] = (
    "confirmed_duplicate_present_in_official_train"
)

test_leakage_reference_df.to_csv(
    INTERIM_ROOT /
    "classification_test_leakage_reference.csv",
    index=False
)


# --------------------------------------------------
# 3. Save the remaining strong but unconfirmed pairs
# --------------------------------------------------

unconfirmed_strong_candidates = strong_candidates[
    ~(
        np.isclose(
            strong_candidates["rmse"],
            0.0,
            atol=1e-12
        )
        &
        np.isclose(
            strong_candidates["correlation"],
            1.0,
            atol=1e-12
        )
    )
].copy()

unconfirmed_strong_candidates.to_csv(
    INTERIM_ROOT /
    "classification_high_similarity_unconfirmed_candidates.csv",
    index=False
)


# --------------------------------------------------
# 4. Print final counts
# --------------------------------------------------

print("FINAL AUDIT EXPORT")
print("-" * 60)

print(
    "Confirmed training exclusions:",
    len(train_exclusion_df)
)

print(
    "Confirmed affected test images:",
    len(test_leakage_reference_df)
)

print(
    "Strong but unconfirmed candidate pairs:",
    len(unconfirmed_strong_candidates)
)

print("\nFiles saved to:")
print(INTERIM_ROOT)

FINAL AUDIT EXPORT
------------------------------------------------------------
Confirmed training exclusions: 104
Confirmed affected test images: 101
Strong but unconfirmed candidate pairs: 21

Files saved to:
C:\Users\HP\Documents\Projects\Brain_Tumor_XAI\data\interim


## Notebook 2 — Final Conclusion

The BRISC dataset archive passed structural, readability, manifest,
checksum, image–mask pairing, and metadata-integrity checks.

However, analysis of the official classification train/test split identified
confirmed cross-split image duplication.

A total of 101 unique test images were confirmed to have duplicate image
content represented within the official training split. This corresponds to
10.10% of the 1,000-image classification test set.

The confirmed contamination was highly class-dependent:

- Meningioma: 96 / 306 test images (31.37%)
- Pituitary: 5 / 300 test images (1.67%)
- Glioma: 0 / 254 test images
- No-tumor: 0 / 140 test images

The corresponding confirmed duplicate set involves 104 unique training
images.

Two forms of duplication were demonstrated:

1. Cross-split files with identical SHA-256 hashes.
2. Files with different SHA-256 hashes but identical decoded RGB pixel
   arrays at their original native dimensions.

For segmentation, exact duplicated MRI inputs were also found across the
official train/test boundary. Their linked masks were not pixel-identical.
Across nine duplicate-MRI mask comparisons, the mean Dice similarity was
0.8648 and the mean IoU was 0.7635, indicating substantial overlap but
non-identical segmentation annotations.

An additional 21 high-similarity classification train/test candidate pairs
remain unconfirmed. These are retained for documentation and are not included
in the automatic exclusion list.

The raw BRISC dataset remains unchanged. Confirmed duplicate training records
have instead been documented in a reproducible exclusion list stored under
`data/interim/`.

Subsequent model-development experiments will use a leakage-controlled
training protocol based only on confirmed duplicate records while preserving
the original BRISC dataset for reproducibility.